# Flickr30k T3→S3 INT8 Full Validation

Kiểm định ứng viên triển khai đã thắng preflight: encoder ONNX FP32 kết hợp
decoder ONNX dynamic per-channel QInt8. Notebook sinh chú thích bằng đúng
Beam-3, length penalty 0,7 cho toàn bộ **1.014 ảnh Karpathy Validation**, so
sánh với ONNX FP32 và đối chiếu checkpoint PyTorch đã khóa. Held-out Test vẫn
niêm phong và tuyệt đối không được đọc.

## Kaggle input bắt buộc — đúng 6 input

1. Output `F30K_T3S3_INT8_Preflight`.
2. Output `F30K_T3S3_ONNX_FP32`.
3. Output `F30K_Data`.
4. Dataset `Flickr Image dataset`.
5. Package/dataset `pycocoevalcap-1.2`.
6. Dataset `ONNX_Runtime_Python313_Offline`.

Không gắn output Held-out Test, MS COCO, checkpoint huấn luyện, Teacher hoặc
seed 123/2026. Khi cần resume, chỉ gắn thêm output gần nhất của chính notebook
`F30K_T3S3_INT8_Validation`.

Runtime: **CPU / Accelerator None / Internet Off**. Dự kiến 45–120 phút tùy
CPU Kaggle; phép đo cục bộ cho thấy riêng hai decoder cần xấp xỉ 1 giây/ảnh.
Notebook lưu tiến độ mỗi 25 ảnh, in log mỗi 10 ảnh và có thể resume chính xác.

Gate đã khóa trước khi chạy: tái lập FP32 trong dung sai đã công bố; độ giảm
CIDEr-D không quá 0,010; BLEU/METEOR/ROUGE-L không giảm quá 0,005; CE không
tăng quá 0,050; caption trùng khớp ít nhất 90%; dung lượng giảm ít nhất 40%.
Nếu chỉ điều kiện trùng nguyên văn thất bại nhưng toàn bộ metric khoa học đạt,
notebook giữ nguyên thất bại strict-gate trong kết quả và chuyển sang paired
bootstrap không-kém-hơn; không tự hạ ngưỡng 90%.

## Cell 1 — Authorization, integrity and runtime

In [1]:
import csv
import hashlib
import importlib.util
import json
import math
import os
import shutil
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

EXPERIMENT_ID = "F30K_T3S3_INT8_Validation"
INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working") / EXPERIMENT_ID
WORK_ROOT.mkdir(parents=True, exist_ok=True)

def sha256(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def write_json_atomic(path, payload):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2) + "\n",
        encoding="utf-8",
    )
    os.replace(temporary, path)

def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (f"Cần đúng một {name}", [str(x) for x in matches])
    return matches[0]

PREFLIGHT_RESULT_PATH = unique_named(
    "F30K_T3S3_INT8_Preflight_RESULT.json"
)
FP32_RESULT_PATH = unique_named("F30K_T3S3_ONNX_FP32_RESULT.json")
DATA_CONTRACT_PATH = unique_named("F30K_DATA_CONTRACT.json")
VAL_PATH = unique_named("f30k_validation.parquet")
VOCAB_PATH = unique_named("f30k_vocabulary_minfreq5.json")

PREFLIGHT = read_json(PREFLIGHT_RESULT_PATH)
FP32_RESULT = read_json(FP32_RESULT_PATH)
DATA_CONTRACT = read_json(DATA_CONTRACT_PATH)
VOCAB = read_json(VOCAB_PATH)

assert PREFLIGHT["status"] == "passed"
assert PREFLIGHT["selected_candidate"] == (
    "mixed_fp32_encoder_dynamic_int8_decoder"
)
assert PREFLIGHT["selected_precision"] == {
    "encoder": "FP32",
    "decoder": "dynamic per-channel QInt8 weights",
}
assert PREFLIGHT["next_phase_authorized"] == EXPERIMENT_ID
assert PREFLIGHT["validation_data_read"] is False
assert PREFLIGHT["test_set_status"] == "sealed"
assert PREFLIGHT["test_set_used"] is False
assert FP32_RESULT["status"] == "passed"
assert FP32_RESULT["source"]["seed"] == 42
assert FP32_RESULT["source"]["method"] == "Direct Logit-KD"
assert DATA_CONTRACT["test_set_status"] == "sealed"
assert DATA_CONTRACT["test_caption_exported"] is False
assert len(VOCAB["tokens"]) == 7456

ENCODER_FP32 = (
    PREFLIGHT_RESULT_PATH.parent / "f30k_t3s3_seed42_encoder_fp32.onnx"
)
DECODER_INT8 = (
    PREFLIGHT_RESULT_PATH.parent
    / "f30k_t3s3_seed42_decoder_dynamic_int8.onnx"
)
DECODER_FP32 = (
    FP32_RESULT_PATH.parent / "f30k_t3s3_seed42_decoder_fp32.onnx"
)
for path, record in [
    (ENCODER_FP32, PREFLIGHT["artifacts"][ENCODER_FP32.name]),
    (DECODER_INT8, PREFLIGHT["artifacts"][DECODER_INT8.name]),
    (DECODER_FP32, FP32_RESULT["artifacts"][DECODER_FP32.name]),
]:
    assert path.is_file(), path
    assert path.stat().st_size == record["bytes"], path
    assert sha256(path) == record["sha256"], path
assert sha256(ENCODER_FP32) == PREFLIGHT["source"]["encoder_sha256"]
assert sha256(DECODER_FP32) == PREFLIGHT["source"]["decoder_sha256"]

for file_name, path in [
    ("f30k_validation.parquet", VAL_PATH),
    ("f30k_vocabulary_minfreq5.json", VOCAB_PATH),
]:
    expected = DATA_CONTRACT["artifacts"][
        "validation" if file_name.endswith(".parquet") else "vocabulary"
    ]["sha256"]
    assert sha256(path) == expected, file_name

if importlib.util.find_spec("onnxruntime") is None:
    wheels = sorted(INPUT_ROOT.rglob("onnxruntime*.whl"))
    assert wheels, "Hãy gắn dataset ONNX_Runtime_Python313_Offline."
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", str(wheels[0])]
    )
if importlib.util.find_spec("pycocoevalcap") is None:
    wheels = sorted(INPUT_ROOT.rglob("pycocoevalcap*.whl"))
    assert wheels, "Hãy gắn package/dataset pycocoevalcap-1.2 offline."
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", str(wheels[0])]
    )
assert shutil.which("java"), "METEOR chính thức yêu cầu Java runtime."

import onnxruntime as ort

providers = ["CPUExecutionProvider"]
session_options = ort.SessionOptions()
session_options.intra_op_num_threads = max(1, os.cpu_count() or 1)
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL
ENCODER_SESSION = ort.InferenceSession(
    str(ENCODER_FP32), sess_options=session_options, providers=providers
)
DECODER_FP32_SESSION = ort.InferenceSession(
    str(DECODER_FP32), sess_options=session_options, providers=providers
)
DECODER_INT8_SESSION = ort.InferenceSession(
    str(DECODER_INT8), sess_options=session_options, providers=providers
)

print("Authorization          : PASSED")
print("Candidate              : FP32 encoder + dynamic INT8 decoder")
print("Source seed / method   : 42 / Direct Logit-KD")
print("Validation authorized  : True")
print("Held-out Test          : SEALED / NOT READ")
print("ONNX Runtime           :", ort.__version__)
print("CPU threads            :", session_options.intra_op_num_threads)

Authorization          : PASSED
Candidate              : FP32 encoder + dynamic INT8 decoder
Source seed / method   : 42 / Direct Logit-KD
Validation authorized  : True
Held-out Test          : SEALED / NOT READ
ONNX Runtime           : 1.28.0
CPU threads            : 4


## Cell 2 — Validation data, preprocessing and Beam-3

In [2]:
val_df = pd.read_parquet(VAL_PATH)
assert val_df["image_id"].nunique() == 1014
assert len(val_df) == 5070

VAL_REFERENCES = {}
VAL_TOKENS = {}
for image_id, group in val_df.groupby("image_id", sort=True):
    iid = int(image_id)
    VAL_REFERENCES[iid] = [str(value) for value in group.caption.tolist()]
    VAL_TOKENS[iid] = [list(map(int, value)) for value in group.token_ids]
assert all(len(values) == 5 for values in VAL_REFERENCES.values())

needed_names = {f"{image_id}.jpg" for image_id in VAL_REFERENCES}
image_candidates = [
    path for path in INPUT_ROOT.rglob("*.jpg") if path.name in needed_names
]
IMAGE_BY_NAME = {}
duplicate_alias_count = 0
for path in image_candidates:
    old = IMAGE_BY_NAME.get(path.name)
    if old is None:
        IMAGE_BY_NAME[path.name] = path
        continue
    assert old.stat().st_size == path.stat().st_size, (
        "Hai ảnh trùng tên nhưng khác kích thước", path.name, old, path
    )
    duplicate_alias_count += 1
    IMAGE_BY_NAME[path.name] = min(
        [old, path],
        key=lambda item: (len(item.parts), len(str(item)), str(item)),
    )
missing = sorted(needed_names - set(IMAGE_BY_NAME))
assert not missing, ("Thiếu ảnh Validation", len(missing), missing[:10])

MEAN = np.asarray([0.485, 0.456, 0.406], dtype=np.float32).reshape(1, 1, 3)
STD = np.asarray([0.229, 0.224, 0.225], dtype=np.float32).reshape(1, 1, 3)
TOKENS = VOCAB["tokens"]
SPECIAL = {0, 1, 2, 3}

def resize_shorter_side(image, target=256):
    width, height = image.size
    if width <= height:
        new_width = target
        new_height = int(target * height / width)
    else:
        new_height = target
        new_width = int(target * width / height)
    return image.resize((new_width, new_height), resample=Image.Resampling.BICUBIC)

def preprocess_image(path):
    with Image.open(path) as opened:
        image = resize_shorter_side(opened.convert("RGB"), 256)
    width, height = image.size
    left = int(round((width - 224) / 2.0))
    top = int(round((height - 224) / 2.0))
    image = image.crop((left, top, left + 224, top + 224))
    array = np.asarray(image, dtype=np.float32) / 255.0
    array = (array - MEAN) / STD
    return np.transpose(array, (2, 0, 1))[None].astype(np.float32)

def ids_to_caption(ids):
    words = []
    for token_id in ids:
        token_id = int(token_id)
        if token_id == 2:
            break
        if token_id not in SPECIAL:
            words.append(TOKENS[token_id])
    text = " ".join(words)
    for punctuation in [".", ",", "!", "?", ";", ":"]:
        text = text.replace(" " + punctuation, punctuation)
    return text.strip()

def stable_log_softmax(values):
    values = np.asarray(values, dtype=np.float64)
    maximum = float(np.max(values))
    shifted = values - maximum
    return shifted - math.log(float(np.exp(shifted).sum()))

def normalized_score(beam, alpha=0.7):
    token_ids, score, _ = beam
    return score / (max(1, len(token_ids) - 1) ** alpha)

def beam_decode(session, memory, beam_size=3, alpha=0.7):
    beams = [([1], 0.0, False)]
    decoder_calls = 0
    for _ in range(30):
        active = [beam for beam in beams if not beam[2]]
        if not active:
            break
        candidates = [beam for beam in beams if beam[2]]
        for token_ids, score, _ in active:
            token_buffer = np.zeros((1, 31), dtype=np.int64)
            attention_mask = np.zeros((1, 31), dtype=np.bool_)
            valid_length = len(token_ids)
            token_buffer[0, :valid_length] = token_ids
            attention_mask[0, :valid_length] = True
            logits = session.run(
                ["logits"],
                {
                    "token_ids": token_buffer,
                    "memory": memory,
                    "attention_mask": attention_mask,
                },
            )[0][0, valid_length - 1].astype(np.float64)
            decoder_calls += 1
            logits[0] = -np.inf
            logits[1] = -np.inf
            log_probabilities = stable_log_softmax(logits)
            top_indices = np.argpartition(
                log_probabilities, -beam_size
            )[-beam_size:]
            top_indices = sorted(
                (int(index) for index in top_indices),
                key=lambda index: (-float(log_probabilities[index]), index),
            )
            for index in top_indices:
                new_tokens = token_ids + [index]
                candidates.append(
                    (
                        new_tokens,
                        score + float(log_probabilities[index]),
                        index == 2,
                    )
                )
        beams = sorted(
            candidates,
            key=lambda beam: (
                -normalized_score(beam, alpha),
                tuple(beam[0]),
            ),
        )[:beam_size]
    best = min(
        beams,
        key=lambda beam: (-normalized_score(beam, alpha), tuple(beam[0])),
    )
    return best[0], decoder_calls

def teacher_forced_ce(session, memory, token_ids):
    sequence = list(map(int, token_ids))[:31]
    assert sequence[0] == 1 and 2 in sequence
    token_buffer = np.zeros((1, 31), dtype=np.int64)
    attention_mask = np.zeros((1, 31), dtype=np.bool_)
    token_buffer[0, :len(sequence)] = sequence
    attention_mask[0, :len(sequence)] = True
    logits = session.run(
        ["logits"],
        {
            "token_ids": token_buffer,
            "memory": memory,
            "attention_mask": attention_mask,
        },
    )[0][0]
    loss_sum = 0.0
    token_count = 0
    for position in range(len(sequence) - 1):
        log_probabilities = stable_log_softmax(logits[position])
        loss_sum -= float(log_probabilities[sequence[position + 1]])
        token_count += 1
    return loss_sum, token_count

print("Validation images      :", len(VAL_REFERENCES))
print("Validation references  :", sum(map(len, VAL_REFERENCES.values())))
print("Vocabulary             :", len(TOKENS))
print("Nested image aliases   :", duplicate_alias_count, "(deduplicated)")
print("Beam / length penalty  : 3 / 0.7")
print("Test                   : SEALED / NOT READ")

Validation images      : 1014
Validation references  : 5070
Vocabulary             : 7456
Nested image aliases   : 1014 (deduplicated)
Beam / length penalty  : 3 / 0.7
Test                   : SEALED / NOT READ


## Cell 3 — Full 1,014-image inference with exact resume

In [3]:
FP32_CAPTIONS_PATH = WORK_ROOT / "fp32_validation_captions.json"
INT8_CAPTIONS_PATH = WORK_ROOT / "mixed_int8_validation_captions.json"
PROGRESS_PATH = WORK_ROOT / f"{EXPERIMENT_ID}_PROGRESS.json"

FP32_CAPTIONS = {}
INT8_CAPTIONS = {}
ce_accumulator = {
    "fp32_loss_sum": 0.0,
    "int8_loss_sum": 0.0,
    "token_count": 0,
}
timing = {
    "encoder_seconds": 0.0,
    "fp32_decoder_seconds": 0.0,
    "int8_decoder_seconds": 0.0,
    "fp32_decoder_calls": 0,
    "int8_decoder_calls": 0,
}

resume_matches = sorted(INPUT_ROOT.rglob(f"{EXPERIMENT_ID}_PROGRESS.json"))
if resume_matches:
    assert len(resume_matches) == 1, resume_matches
    resume_progress = read_json(resume_matches[0])
    assert resume_progress["experiment_id"] == EXPERIMENT_ID
    assert resume_progress["preflight_result_sha256"] == sha256(
        PREFLIGHT_RESULT_PATH
    )
    assert resume_progress["validation_sha256"] == sha256(VAL_PATH)
    source_directory = resume_matches[0].parent
    FP32_CAPTIONS = {
        int(key): value
        for key, value in read_json(
            source_directory / FP32_CAPTIONS_PATH.name
        ).items()
    }
    INT8_CAPTIONS = {
        int(key): value
        for key, value in read_json(
            source_directory / INT8_CAPTIONS_PATH.name
        ).items()
    }
    ce_accumulator = resume_progress["ce_accumulator"]
    timing = resume_progress["timing"]
    assert set(FP32_CAPTIONS) == set(INT8_CAPTIONS)
    print("Exact resume           :", len(FP32_CAPTIONS), "/ 1014 images")

def save_progress():
    write_json_atomic(
        FP32_CAPTIONS_PATH,
        {str(key): FP32_CAPTIONS[key] for key in sorted(FP32_CAPTIONS)},
    )
    write_json_atomic(
        INT8_CAPTIONS_PATH,
        {str(key): INT8_CAPTIONS[key] for key in sorted(INT8_CAPTIONS)},
    )
    write_json_atomic(
        PROGRESS_PATH,
        {
            "schema_version": "f30k_t3s3_int8_validation_progress_v1",
            "experiment_id": EXPERIMENT_ID,
            "completed_images": len(FP32_CAPTIONS),
            "total_images": len(VAL_REFERENCES),
            "preflight_result_sha256": sha256(PREFLIGHT_RESULT_PATH),
            "validation_sha256": sha256(VAL_PATH),
            "ce_accumulator": ce_accumulator,
            "timing": timing,
            "test_set_status": "sealed",
            "test_set_used": False,
        },
    )

run_started = time.perf_counter()
all_image_ids = sorted(VAL_REFERENCES)
for ordinal, image_id in enumerate(all_image_ids, start=1):
    if image_id in FP32_CAPTIONS:
        continue
    image = preprocess_image(IMAGE_BY_NAME[f"{image_id}.jpg"])
    started = time.perf_counter()
    memory = ENCODER_SESSION.run(["memory"], {"images": image})[0]
    timing["encoder_seconds"] += time.perf_counter() - started
    assert memory.shape == (1, 49, 256)
    assert np.isfinite(memory).all()

    started = time.perf_counter()
    fp32_ids, fp32_calls = beam_decode(DECODER_FP32_SESSION, memory)
    fp32_loss_sum, fp32_token_count = teacher_forced_ce(
        DECODER_FP32_SESSION, memory, VAL_TOKENS[image_id][0]
    )
    timing["fp32_decoder_seconds"] += time.perf_counter() - started
    timing["fp32_decoder_calls"] += fp32_calls + 1

    started = time.perf_counter()
    int8_ids, int8_calls = beam_decode(DECODER_INT8_SESSION, memory)
    int8_loss_sum, int8_token_count = teacher_forced_ce(
        DECODER_INT8_SESSION, memory, VAL_TOKENS[image_id][0]
    )
    timing["int8_decoder_seconds"] += time.perf_counter() - started
    timing["int8_decoder_calls"] += int8_calls + 1
    assert fp32_token_count == int8_token_count

    FP32_CAPTIONS[image_id] = ids_to_caption(fp32_ids)
    INT8_CAPTIONS[image_id] = ids_to_caption(int8_ids)
    ce_accumulator["fp32_loss_sum"] += fp32_loss_sum
    ce_accumulator["int8_loss_sum"] += int8_loss_sum
    ce_accumulator["token_count"] += fp32_token_count

    completed = len(FP32_CAPTIONS)
    if completed % 10 == 0 or completed == len(all_image_ids):
        elapsed = time.perf_counter() - run_started
        processed_this_run = sum(
            1 for key in FP32_CAPTIONS if key in set(all_image_ids)
        )
        print(
            f"Validation {completed:04d}/{len(all_image_ids)} | "
            f"current image={image_id} | run elapsed={elapsed / 60:.1f} min",
            flush=True,
        )
    if completed % 25 == 0:
        save_progress()

save_progress()
assert len(FP32_CAPTIONS) == 1014
assert len(INT8_CAPTIONS) == 1014
assert set(FP32_CAPTIONS) == set(VAL_REFERENCES)
assert ce_accumulator["token_count"] > 0
print("Inference             : COMPLETE")
print("Captions FP32 / INT8  :", len(FP32_CAPTIONS), "/", len(INT8_CAPTIONS))
print("Held-out Test         : SEALED / NOT READ")

Validation 0010/1014 | current image=10350842 | run elapsed=0.2 min
Validation 0020/1014 | current image=44953142 | run elapsed=0.3 min
Validation 0030/1014 | current image=78171801 | run elapsed=0.5 min
Validation 0040/1014 | current image=109982467 | run elapsed=0.6 min
Validation 0050/1014 | current image=134642833 | run elapsed=0.8 min
Validation 0060/1014 | current image=146758244 | run elapsed=1.0 min
Validation 0070/1014 | current image=180217379 | run elapsed=1.2 min
Validation 0080/1014 | current image=213609234 | run elapsed=1.3 min
Validation 0090/1014 | current image=241347611 | run elapsed=1.5 min
Validation 0100/1014 | current image=275854404 | run elapsed=1.7 min
Validation 0110/1014 | current image=314454989 | run elapsed=1.9 min
Validation 0120/1014 | current image=365849741 | run elapsed=2.0 min
Validation 0130/1014 | current image=384080245 | run elapsed=2.2 min
Validation 0140/1014 | current image=416933689 | run elapsed=2.3 min
Validation 0150/1014 | current image=

## Cell 4 — Official metrics and predeclared gates

In [4]:
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

def official_metrics(captions):
    references = {
        int(key): [{"caption": value} for value in values]
        for key, values in VAL_REFERENCES.items()
    }
    hypotheses = {
        int(key): [{"caption": captions[int(key)]}]
        for key in VAL_REFERENCES
    }
    tokenizer = PTBTokenizer()
    references = tokenizer.tokenize(references)
    hypotheses = tokenizer.tokenize(hypotheses)
    bleu, _ = Bleu(4).compute_score(references, hypotheses)
    meteor = Meteor()
    try:
        meteor_score, _ = meteor.compute_score(references, hypotheses)
    finally:
        close_method = getattr(meteor, "close", None)
        if callable(close_method):
            close_method()
    rouge, _ = Rouge().compute_score(references, hypotheses)
    cider, _ = Cider().compute_score(references, hypotheses)
    return {
        "BLEU_1": float(bleu[0]),
        "BLEU_4": float(bleu[3]),
        "METEOR": float(meteor_score),
        "ROUGE_L": float(rouge),
        "CIDEr_D": float(cider),
    }

FP32_METRICS = official_metrics(FP32_CAPTIONS)
INT8_METRICS = official_metrics(INT8_CAPTIONS)
FP32_METRICS["ce_loss"] = (
    ce_accumulator["fp32_loss_sum"] / ce_accumulator["token_count"]
)
INT8_METRICS["ce_loss"] = (
    ce_accumulator["int8_loss_sum"] / ce_accumulator["token_count"]
)

LOCKED_PYTORCH = {
    "BLEU_1": 0.6952422907488374,
    "BLEU_4": 0.279519800720964,
    "METEOR": 0.22151968028521943,
    "ROUGE_L": 0.49543244311706297,
    "CIDEr_D": 0.617055344048548,
    "ce_loss": 2.7545091490472857,
}
REPRODUCTION_TOLERANCE = {
    "BLEU_1": 0.005,
    "BLEU_4": 0.005,
    "METEOR": 0.005,
    "ROUGE_L": 0.005,
    "CIDEr_D": 0.010,
    "ce_loss": 0.010,
}
reproduction_gates = {}
for metric, expected in LOCKED_PYTORCH.items():
    difference = FP32_METRICS[metric] - expected
    tolerance = REPRODUCTION_TOLERANCE[metric]
    reproduction_gates[metric] = {
        "locked_pytorch": expected,
        "onnx_fp32": FP32_METRICS[metric],
        "difference": difference,
        "absolute_tolerance": tolerance,
        "passed": abs(difference) <= tolerance,
    }

QUALITY_DROP_LIMITS = {
    "BLEU_1": 0.005,
    "BLEU_4": 0.005,
    "METEOR": 0.005,
    "ROUGE_L": 0.005,
    "CIDEr_D": 0.010,
    "ce_loss_increase": 0.050,
}
quality_gates = {}
for metric in ["BLEU_1", "BLEU_4", "METEOR", "ROUGE_L", "CIDEr_D"]:
    delta = INT8_METRICS[metric] - FP32_METRICS[metric]
    quality_gates[metric] = {
        "fp32": FP32_METRICS[metric],
        "mixed_int8": INT8_METRICS[metric],
        "delta": delta,
        "maximum_allowed_drop": QUALITY_DROP_LIMITS[metric],
        "passed": delta >= -QUALITY_DROP_LIMITS[metric],
    }
ce_increase = INT8_METRICS["ce_loss"] - FP32_METRICS["ce_loss"]
quality_gates["ce_loss"] = {
    "fp32": FP32_METRICS["ce_loss"],
    "mixed_int8": INT8_METRICS["ce_loss"],
    "increase": ce_increase,
    "maximum_allowed_increase": QUALITY_DROP_LIMITS["ce_loss_increase"],
    "passed": ce_increase <= QUALITY_DROP_LIMITS["ce_loss_increase"],
}

exact_caption_count = sum(
    FP32_CAPTIONS[key] == INT8_CAPTIONS[key] for key in FP32_CAPTIONS
)
exact_caption_agreement = exact_caption_count / len(FP32_CAPTIONS)
agreement_gate = {
    "exact_caption_count": exact_caption_count,
    "total": len(FP32_CAPTIONS),
    "agreement": exact_caption_agreement,
    "minimum_required": 0.90,
    "passed": exact_caption_agreement >= 0.90,
}
size_gate = {
    "fp32_reference_MiB": PREFLIGHT["selected_size"]["fp32_reference_MiB"],
    "mixed_int8_MiB": PREFLIGHT["selected_size"]["selected_MiB"],
    "reduction_fraction": PREFLIGHT["selected_size"]["reduction_fraction"],
    "minimum_required": 0.40,
    "passed": PREFLIGHT["selected_size"]["reduction_fraction"] >= 0.40,
}

reproduction_passed = all(
    record["passed"] for record in reproduction_gates.values()
)
quality_passed = all(record["passed"] for record in quality_gates.values())
strict_all_passed = (
    reproduction_passed
    and quality_passed
    and agreement_gate["passed"]
    and size_gate["passed"]
)
scientific_quality_passed = (
    reproduction_passed and quality_passed and size_gate["passed"]
)

print("FP32 metrics          :", FP32_METRICS)
print("Mixed INT8 metrics    :", INT8_METRICS)
print("Exact agreement       :", exact_caption_agreement)
print("FP32 reproduction     :", "PASSED" if reproduction_passed else "FAILED")
print("INT8 quality gate     :", "PASSED" if quality_passed else "FAILED")
print("Size gate             :", "PASSED" if size_gate["passed"] else "FAILED")
print("Strict verbatim gate  :", "PASSED" if strict_all_passed else "FAILED")
print("Scientific quality    :",
      "PASSED" if scientific_quality_passed else "FAILED")

PTBTokenizer tokenized 74030 tokens at 232617.36 tokens per second.
PTBTokenizer tokenized 13429 tokens at 101537.56 tokens per second.


{'testlen': 11361, 'reflen': 11092, 'guess': [11361, 10347, 9333, 8319], 'correct': [7891, 4000, 1912, 910]}
ratio: 1.0242517129461752


PTBTokenizer tokenized 74030 tokens at 269308.71 tokens per second.
PTBTokenizer tokenized 13381 tokens at 94405.34 tokens per second.


{'testlen': 11321, 'reflen': 11090, 'guess': [11321, 10307, 9293, 8279], 'correct': [7863, 4000, 1919, 914]}
ratio: 1.020829576194678
FP32 metrics          : {'BLEU_1': 0.6945691400404282, 'BLEU_4': 0.27851557012947864, 'METEOR': 0.221282026466254, 'ROUGE_L': 0.4955252831934373, 'CIDEr_D': 0.6145264982944407, 'ce_loss': 2.754514845508431}
Mixed INT8 metrics    : {'BLEU_1': 0.6945499514176579, 'BLEU_4': 0.2799816946180795, 'METEOR': 0.22238177323093886, 'ROUGE_L': 0.4956929458524092, 'CIDEr_D': 0.6171285226440146, 'ce_loss': 2.7555685628753697}
Exact agreement       : 0.7564102564102564
FP32 reproduction     : PASSED
INT8 quality gate     : PASSED
Size gate             : PASSED
Strict verbatim gate  : FAILED
Scientific quality    : PASSED


## Cell 5 — Results, manifest and deployment authorization

In [5]:
rows = []
for image_id in sorted(VAL_REFERENCES):
    rows.append({
        "image_id": image_id,
        "fp32_caption": FP32_CAPTIONS[image_id],
        "mixed_int8_caption": INT8_CAPTIONS[image_id],
        "exact_match": FP32_CAPTIONS[image_id] == INT8_CAPTIONS[image_id],
        "reference_1": VAL_REFERENCES[image_id][0],
        "reference_2": VAL_REFERENCES[image_id][1],
        "reference_3": VAL_REFERENCES[image_id][2],
        "reference_4": VAL_REFERENCES[image_id][3],
        "reference_5": VAL_REFERENCES[image_id][4],
    })
captions_csv = WORK_ROOT / f"{EXPERIMENT_ID}_CAPTIONS.csv"
pd.DataFrame(rows).to_csv(captions_csv, index=False, encoding="utf-8")

result = {
    "schema_version": "f30k_t3s3_int8_validation_quality_v2",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": (
        "passed"
        if strict_all_passed
        else (
            "passed_with_caption_variation_review_required"
            if scientific_quality_passed
            else "failed"
        )
    ),
    "experiment_id": EXPERIMENT_ID,
    "pair": "T3 SigLIP ViT-B/16 adapted -> S3 TinyViT-5M",
    "method": "Direct Logit-KD",
    "seed": 42,
    "evaluation_split": "Flickr30k Karpathy Validation",
    "validation_images": len(VAL_REFERENCES),
    "validation_references": sum(map(len, VAL_REFERENCES.values())),
    "decoding": {"beam_size": 3, "length_penalty": 0.7, "max_steps": 30},
    "precision": {
        "reference": "ONNX FP32 encoder + FP32 decoder",
        "candidate": "ONNX FP32 encoder + dynamic per-channel QInt8 decoder",
    },
    "metrics": {
        "onnx_fp32": FP32_METRICS,
        "mixed_int8": INT8_METRICS,
        "mixed_minus_fp32": {
            metric: INT8_METRICS[metric] - FP32_METRICS[metric]
            for metric in FP32_METRICS
        },
    },
    "gates": {
        "fp32_reproduction": reproduction_gates,
        "mixed_int8_quality": quality_gates,
        "caption_agreement": agreement_gate,
        "size": size_gate,
    },
    "gate_interpretation": {
        "strict_original_gate_passed": strict_all_passed,
        "scientific_quality_gate_passed": scientific_quality_passed,
        "caption_agreement_role": (
            "Diagnostic verbatim-stability indicator; failure cannot be hidden "
            "or relabeled as strict pass. A paired image-level CIDEr-D "
            "non-inferiority review is required before deployment lock."
        ),
    },
    "timing": timing,
    "source_integrity": {
        "preflight_result_sha256": sha256(PREFLIGHT_RESULT_PATH),
        "fp32_result_sha256": sha256(FP32_RESULT_PATH),
        "data_contract_sha256": sha256(DATA_CONTRACT_PATH),
        "validation_sha256": sha256(VAL_PATH),
        "vocabulary_sha256": sha256(VOCAB_PATH),
        "encoder_fp32_sha256": sha256(ENCODER_FP32),
        "decoder_fp32_sha256": sha256(DECODER_FP32),
        "decoder_int8_sha256": sha256(DECODER_INT8),
    },
    "validation_data_read": True,
    "held_out_test_authorized": False,
    "test_set_status": "sealed",
    "test_set_used": False,
    "next_phase_authorized": (
        "F30K_T3S3_Deployment_Lock"
        if strict_all_passed
        else (
            "F30K_T3S3_INT8_NonInferiority"
            if scientific_quality_passed
            else "STOP_REVIEW_INT8_QUALITY_FAILURE"
        )
    ),
}
result_path = WORK_ROOT / f"{EXPERIMENT_ID}_RESULT.json"
write_json_atomic(result_path, result)

manifest_files = [
    FP32_CAPTIONS_PATH,
    INT8_CAPTIONS_PATH,
    PROGRESS_PATH,
    captions_csv,
    result_path,
]
manifest = {
    "schema_version": "f30k_t3s3_int8_validation_manifest_v1",
    "experiment_id": EXPERIMENT_ID,
    "status": "complete",
    "artifacts": [
        {
            "relative_path": path.name,
            "size_bytes": path.stat().st_size,
            "sha256": sha256(path),
        }
        for path in manifest_files
    ],
    "validation_data_read": True,
    "test_set_status": "sealed",
    "test_set_used": False,
}
write_json_atomic(WORK_ROOT / "MANIFEST.json", manifest)

print("Status                :", result["status"].upper())
print("FP32 CIDEr-D          :", FP32_METRICS["CIDEr_D"])
print("Mixed INT8 CIDEr-D    :", INT8_METRICS["CIDEr_D"])
print("CIDEr-D delta         :",
      INT8_METRICS["CIDEr_D"] - FP32_METRICS["CIDEr_D"])
print("Package MiB           :", size_gate["mixed_int8_MiB"])
print("Size reduction        :", size_gate["reduction_fraction"])
print("Held-out Test         : SEALED / NOT USED")
print("Next                  :", result["next_phase_authorized"])
print("Output                :", WORK_ROOT)
assert scientific_quality_passed, result["gates"]

Status                : PASSED_WITH_CAPTION_VARIATION_REVIEW_REQUIRED
FP32 CIDEr-D          : 0.6145264982944407
Mixed INT8 CIDEr-D    : 0.6171285226440146
CIDEr-D delta         : 0.002602024349573906
Package MiB           : 27.182700157165527
Size reduction        : 0.42130889789589887
Held-out Test         : SEALED / NOT USED
Next                  : F30K_T3S3_INT8_NonInferiority
Output                : /kaggle/working/F30K_T3S3_INT8_Validation
